# Kochi-LLM Phase 2 — finish to exactly one epoch
Attach ONLY the immediately-preceding Phase-2 output. It supplies the checkpoint, tokenizer, data bins, and `.py` files. Internet OFF, GPU T4 x2.
Schedule is now 5500 iters = 180,224,000 tokens = 0.9996 epochs of the 180,295,081-token train split, so cosine anneals fully (LR ends at min_lr).
The setup picks the newest `latest_backup_model.pth` and restarts Phase-2 best tracking, so the stale Alpaca score of 3.8501 can no longer block saves.
~3 sessions left. Each run: attach only the previous run's output, Save & Run.

In [ ]:
%%bash
set -e
LATEST=""
LATEST_T=0
while IFS= read -r f; do
  [ -z "$f" ] && continue
  t=$(stat -c %Y "$f")
  if [ "$t" -gt "$LATEST_T" ]; then LATEST_T=$t; LATEST=$f; fi
done <<EOF
$(find /kaggle/input -name latest_backup_model.pth 2>/dev/null)
EOF
[ -z "$LATEST" ] && { echo "NO latest_backup_model.pth ATTACHED"; ls -R /kaggle/input | head -40; exit 1; }
CKPT_SRC=$(dirname "$LATEST")
echo "CKPT_SRC=$CKPT_SRC"
for f in train.bin val.bin custom_bpe_tokenizer.pkl; do
  [ -f "$CKPT_SRC/$f" ] || { echo "MISSING $f"; exit 1; }
done
PY_SRC=""
for d in /kaggle/input/datasets/dadhichisarker/kochi-llm/model "$CKPT_SRC"; do
  if [ -f "$d/architecture.py" ]; then PY_SRC="$d"; break; fi
done
[ -z "$PY_SRC" ] && { echo "NO PY SOURCE"; ls -R /kaggle/input | head -40; exit 1; }
echo "PY_SRC=$PY_SRC"
SZ=$(stat -c %s "$CKPT_SRC/train.bin")
[ "$SZ" -lt 100000000 ] && { echo "train.bin too small ($SZ bytes)"; exit 1; }
cp "$PY_SRC"/*.py /kaggle/working/
cp "$CKPT_SRC"/latest_backup_model.pth /kaggle/working/resume_model.pth
cp "$CKPT_SRC"/train.bin "$CKPT_SRC"/val.bin "$CKPT_SRC"/custom_bpe_tokenizer.pkl /kaggle/working/
rm -f /kaggle/working/best_model.pth /kaggle/working/latest_backup_model.pth
echo "Seeded from newest backup; Phase-2 best tracking will restart."
ls -lh /kaggle/working/*.py /kaggle/working/*.pth /kaggle/working/*.bin /kaggle/working/*.pkl


In [ ]:
%%writefile train_custom.py
# Phase 2 (cont.): finish to one epoch (5500 iters), seeds from previous latest_backup via setup cell.
import torch, time, math, os, functools, warnings
warnings.filterwarnings('ignore', message='.*FSDP.state_dict_type.*')
from architecture import MyCustomLLM, CustomBlock
from data_loader import CustomDataLoader

batch_size = 16
micro_batch_size = 1
gradient_accumulation_steps = batch_size // micro_batch_size
block_size = 1024
bpe_vocab_size = 32000

distributed = int(os.environ.get('RANK', -1)) != -1
if distributed:
    import torch.distributed as dist
    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
    from torch.distributed.fsdp import MixedPrecision, ShardingStrategy
    from torch.distributed.fsdp import FullStateDictConfig, StateDictType
    from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
    dist.init_process_group(backend='nccl')
    local_rank = int(os.environ['LOCAL_RANK'])
    torch.cuda.set_device(local_rank)
    device = f'cuda:{local_rank}'
    is_main = (local_rank == 0)
    if is_main: print(f"FSDP: {dist.get_world_size()} GPUs")
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    local_rank = 0
    is_main = True

device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

train_dataset = CustomDataLoader(block_size, split='train', vocab_size=bpe_vocab_size)
val_dataset = CustomDataLoader(block_size, split='val', vocab_size=bpe_vocab_size)
vocab_size = train_dataset.tokenizer.vocab_size

model_config = dict(vocab_size=vocab_size, dim=1536, n_layers=12, n_heads=24,
    n_kv_heads=8, max_seq_len=block_size, num_experts=4, top_k=2,
    dropout=0.05, window_size=512, n_predict=2)
model = MyCustomLLM(**model_config)
model.to(device)
full_param_count = sum(p.numel() for p in model.parameters())

if distributed:
    auto_wrap_policy = functools.partial(transformer_auto_wrap_policy,
        transformer_layer_cls={CustomBlock})
    mp_policy = MixedPrecision(param_dtype=ptdtype, reduce_dtype=ptdtype, buffer_dtype=ptdtype)
    model = FSDP(model, auto_wrap_policy=auto_wrap_policy, mixed_precision=mp_policy,
        sharding_strategy=ShardingStrategy.FULL_SHARD, device_id=torch.device(device),
        limit_all_gathers=True)
    if is_main: print("FSDP FULL_SHARD active")

if is_main:
    print(f"{full_param_count/1e6:.0f}M params | {model_config['n_layers']}L / {model_config['n_heads']}H / "
          f"{model_config['num_experts']}E+shared-top{model_config['top_k']} / predict-{model_config['n_predict']}")

max_iters, eval_interval, eval_iters, grad_clip, time_limit_hours = 5500, 200, 10, 1.0, 10
max_lr, min_lr, warmup_iters = 1e-3, 1e-4, 200

def get_lr(it):
    if it < warmup_iters: return max_lr * (it + 1) / warmup_iters
    if it > max_iters: return min_lr
    return min_lr + 0.5 * (1 + math.cos(math.pi * (it - warmup_iters) / (max_iters - warmup_iters))) * (max_lr - min_lr)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for name, ds in [('train', train_dataset), ('val', val_dataset)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = ds.get_batch(micro_batch_size, device)
            with torch.autocast(device_type=device_type, dtype=ptdtype):
                _, loss = model(X, Y)
                if loss.dim() > 0: loss = loss.mean()
            losses[k] = loss.item()
        out[name] = losses.mean()
    model.train()
    return out

def save_checkpoint(model, path, config, iter_num, best_val_loss, latest_val_loss):
    if distributed:
        cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
        with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, cfg):
            sd = model.state_dict()
        sd = {k.replace('.module.', '.'): v for k, v in sd.items()}
    else:
        sd = model.state_dict()
    if is_main:
        torch.save({'model_state_dict': sd, 'config': config, 'iter': iter_num,
                   'val_loss': best_val_loss, 'phase2_best_val_loss': best_val_loss,
                   'latest_val_loss': latest_val_loss}, path)

def load_checkpoint(model, path):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    sd = ckpt['model_state_dict']
    sd = {k.replace('.module.', '.'): v for k, v in sd.items()}
    if distributed:
        cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=False)
        with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, cfg):
            model.load_state_dict(sd)
    else:
        model.load_state_dict(sd)
    return (ckpt.get('iter', -1) + 1,
            ckpt.get('phase2_best_val_loss', float('inf')),
            ckpt.get('latest_val_loss', float('inf')))

start_iter, best_val_loss, latest_val_loss = 0, float('inf'), float('inf')
if os.path.exists('resume_model.pth'):
    if is_main: print("Resuming...")
    start_iter, best_val_loss, latest_val_loss = load_checkpoint(model, 'resume_model.pth')
    if is_main:
        print(f"Step {start_iter-1} | Phase-2 best {best_val_loss:.4f} | latest {latest_val_loss:.4f}")

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.1)
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == 'float16'))

t0 = time.time()
completed = True
for it in range(start_iter, max_iters):
    lr = get_lr(it)
    for pg in optimizer.param_groups: pg['lr'] = lr
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for ms in range(gradient_accumulation_steps):
        X, Y = train_dataset.get_batch(micro_batch_size, device)
        with torch.autocast(device_type=device_type, dtype=ptdtype):
            _, loss = model(X, Y)
            if loss.dim() > 0: loss = loss.mean()
            loss = loss / gradient_accumulation_steps
        if distributed and ms < gradient_accumulation_steps - 1:
            with model.no_sync(): scaler.scale(loss).backward()
        else:
            scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    model.clip_grad_norm_(grad_clip) if distributed else torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        train_loss, val_loss = float(losses['train']), float(losses['val'])
        should_save = 0
        if is_main:
            latest_val_loss = val_loss
            print(f"Step {it:4d} | {time.time()-t0:6.0f}s | LR {lr:.2e} | train {train_loss:.4f} | val {val_loss:.4f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                should_save = 1
        if distributed:
            ss = torch.tensor([should_save], device=device)
            dist.broadcast(ss, src=0)
            should_save = int(ss.item())
            bv = torch.tensor([best_val_loss if is_main else 0.0], device=device)
            dist.broadcast(bv, src=0)
            best_val_loss = float(bv.item())
            lv = torch.tensor([latest_val_loss if is_main else 0.0], device=device)
            dist.broadcast(lv, src=0)
            latest_val_loss = float(lv.item())
        if should_save:
            save_checkpoint(model, 'best_model.pth', model_config, it, best_val_loss, latest_val_loss)
            if distributed: dist.barrier()

    time_up = 1 if (time.time() - t0) / 3600 >= time_limit_hours else 0
    if distributed:
        tu = torch.tensor([time_up if is_main else 0], device=device)
        dist.broadcast(tu, src=0)
        time_up = int(tu.item())
    if time_up:
        if is_main: print(f"\n[TIME CAP] Saving...")
        save_checkpoint(model, 'latest_backup_model.pth', model_config, it, best_val_loss, latest_val_loss)
        if distributed: dist.barrier()
        completed = False
        break

if completed:
    save_checkpoint(model, 'latest_backup_model.pth', model_config, max_iters - 1, best_val_loss, latest_val_loss)
    if distributed: dist.barrier()

if is_main: print(f"\nDone in {time.time()-t0:.0f}s | Phase-2 best val: {best_val_loss:.4f}")
if distributed: dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_custom.py